In [1]:
!pip install imblearn


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from collections import Counter
from sklearn.metrics import confusion_matrix
from xgboost import XGBClassifier
from joblib import Memory
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV






plt.rc('font', family='Malgun Gothic')
plt.rc('axes', unicode_minus=False)

# 1. 데이터 로드 + 기본 탐색 + 파생변수 생성

In [3]:
df = pd.read_csv('./data/steam_reviews_last365d.csv')

C:\Users\user\AppData\Local\Temp\ipykernel_20300\3282841514.py:1: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('./data/steam_reviews_last365d.csv')


In [4]:
df.columns

Index(['appid', 'recommendationid', 'steamid', 'num_games_owned',
       'num_reviews_author', 'playtime_forever', 'playtime_last_two_weeks',
       'playtime_at_review', 'deck_playtime_at_review', 'last_played',
       'language', 'review', 'timestamp_created', 'timestamp_updated',
       'voted_up', 'votes_up', 'votes_funny', 'weighted_vote_score',
       'comment_count', 'steam_purchase', 'received_for_free',
       'written_during_early_access', 'developer_response',
       'timestamp_dev_responded', 'primarily_steam_deck'],
      dtype='object')

| 컬럼명(한글) | 컬럼명(영어) | 설명 | 분포(실제 분포) |
|---|---|---|---|
| 게임 ID | appid | Steam 게임 고유 ID | 440 ~ 3,241,660 (mean≈1,239,968) |
| 리뷰 추천 ID | recommendationid | 리뷰 고유 식별자 | 약 1.99e8 ~ 2.15e8 |
| 유저 Steam ID | steamid | 리뷰 작성자 Steam ID | 거의 단일값, 분산 매우 작음 |
| 보유 게임 수 | num_games_owned | 유저가 보유한 전체 게임 수 | 0 ~ 7,706 (median 0, mean 55.9) |
| 작성 리뷰 수 | num_reviews_author | 유저가 작성한 전체 리뷰 수 | 1 ~ 2,542 (median 3) |
| 누적 플레이 타임 | playtime_forever | 해당 게임 총 플레이 시간 | 5 ~ 1,457,369 |
| 최근 2주 플레이 타임 | playtime_last_two_weeks | 최근 2주간 플레이 시간 | 0 ~ 17,144 (75% ≤ 370) |
| 리뷰 시점 플레이 타임 | playtime_at_review | 리뷰 작성 시점 누적 플레이 시간 | 5 ~ 1,396,679 |
| Steam Deck 플레이 타임 | deck_playtime_at_review | 리뷰 시점 Steam Deck 플레이 시간 | 1 ~ 51,724 (표본 적음) |
| 마지막 플레이 시각 | last_played | 마지막 플레이 시점 (Unix) | 1.47e9 ~ 1.77e9 |
| 리뷰 생성 시각 | timestamp_created | 리뷰 최초 작성 시각 (Unix) | 1.75e9 ~ 1.77e9 |
| 리뷰 수정 시각 | timestamp_updated | 리뷰 최종 수정 시각 (Unix) | 생성 시각과 거의 동일 |
| 긍정 추천 여부 | voted_up | 긍정 리뷰 여부 (1=긍정) | mean 0.633 (긍정 약 63%) |
| 도움됨 투표 수 | votes_up | 도움이 됐다고 평가한 수 | 0 ~ 1,511 (대부분 0) |
| 재미있음 투표 수 | votes_funny | 재미있다고 평가한 수 | 0 ~ 438 (대부분 0) |
| 가중 투표 점수 | weighted_vote_score | Steam 내부 도움도 점수 | 0.29 ~ 0.95 (median 0.5) |
| 댓글 수 | comment_count | 리뷰에 달린 댓글 수 | 0 ~ 14 (99% 이상 0) |
| 개발자 응답 시각 | timestamp_dev_responded | 개발자 답변 시각 (Unix) | 존재 데이터 56건 |
| Steam 구매 여부 | steam_purchase | Steam에서 구매했는지 여부 | 없음 |
| 무료 획득 여부 | received_for_free | 무료로 받았는지 여부 | 없음 |
| 얼리액세스 작성 여부 | written_during_early_access | 얼리액세스 중 작성 여부 | 없음 |
| 리뷰 언어 | language | 리뷰 작성 언어 | 없음 |
| 리뷰 텍스트 | review | 리뷰 본문 텍스트 | 없음 |
| 개발자 응답 내용 | developer_response | 개발자 답변 텍스트 | 없음 |
| Steam Deck 주 사용 여부 | primarily_steam_deck | Steam Deck 위주 플레이 여부 | 없음 |
| 중복 appid | appid_1 | appid 중복 컬럼 | appid와 동일 |
| 난수 컬럼 | rnd | 무작위 샘플링용 컬럼 | 0 ~ 0.019 | 

In [5]:
df# 상위 50개 게임으로만 이루어진 데이터
appid_counts = df['appid'].value_counts()
top50_appids = appid_counts.head(50).index.tolist()
df_top50 = df[df['appid'].isin(top50_appids)].copy()

# 결측치 확인
df_top50.isnull().sum()

appid                                0
recommendationid                     0
steamid                              0
num_games_owned                      0
num_reviews_author                   0
playtime_forever                     0
playtime_last_two_weeks              0
playtime_at_review                   0
deck_playtime_at_review        4736661
last_played                          0
language                             0
review                           15761
timestamp_created                    0
timestamp_updated                    0
voted_up                             0
votes_up                             0
votes_funny                          0
weighted_vote_score                  0
comment_count                        0
steam_purchase                       0
received_for_free                    0
written_during_early_access          0
developer_response             4824716
timestamp_dev_responded        4824716
primarily_steam_deck                 0
dtype: int64

In [6]:
# 결측치 있는 컬럼들 모두 제거
df_model = df_top50.drop(columns=['deck_playtime_at_review', 'developer_response', 'timestamp_dev_responded'])
df_model.isnull().sum()

appid                              0
recommendationid                   0
steamid                            0
num_games_owned                    0
num_reviews_author                 0
playtime_forever                   0
playtime_last_two_weeks            0
playtime_at_review                 0
last_played                        0
language                           0
review                         15761
timestamp_created                  0
timestamp_updated                  0
voted_up                           0
votes_up                           0
votes_funny                        0
weighted_vote_score                0
comment_count                      0
steam_purchase                     0
received_for_free                  0
written_during_early_access        0
primarily_steam_deck               0
dtype: int64

In [7]:
df_model

,appid,recommendationid,steamid,num_games_owned,num_reviews_author,playtime_forever,playtime_last_two_weeks,playtime_at_review,last_played,language,...,timestamp_updated,voted_up,votes_up,votes_funny,weighted_vote_score,comment_count,steam_purchase,received_for_free,written_during_early_access,primarily_steam_deck
27798,2139460,215256415,76561198092089560,0,1,1659,1659,1628.0,1767647101,english,...,1767645356,True,0,0,0.50000,0,False,False,False,False
27799,2139460,215256182,76561197995642012,0,4,370,367,339.0,1767646994,french,...,1767645190,True,0,0,0.50000,0,False,False,False,False
27800,2139460,215249671,76561198217416651,0,32,552,552,401.0,1767648127,greek,...,1767640383,True,0,0,0.50000,0,False,False,False,False
27801,2139460,215246874,76561198111964424,0,16,146,0,146.0,1765836608,russian,...,1767638383,False,0,0,0.50000,0,False,False,False,False
27802,2139460,215244452,76561198000529800,85,1,47556,276,47519.0,1767638299,english,...,1767636538,True,0,0,0.50000,0,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5447568,570,192652367,76561199824228518,79,7,49329,9264,5671.0,1767667012,russian,...,1757669172,True,0,0,0.50000,0,False,True,False,False
5447569,570,192652315,76561199509986279,0,1,129565,591,79079.0,1767661292,russian,...,1744542216,False,0,0,0.50000,0,False,False,False,False
5447570,570,192652264,76561199699553706,21,4,36710,548,28793.0,1767315372,russian,...,1762351962,True,0,0,0.50000,0,False,False,False,False
5447571,570,192652232,76561199815094533,2,1,3633,0,1422.0,1766312795,russian,...,1744542138,True,1,0,0.52381,1,False,False,False,False


### game_stlye 매핑 + none 기본값

In [8]:
# Top50 기반으로 매핑 (네가 적어준 game_style 그대로 반영)
# ※ 기존 29개에 없던 appid도 포함해서 업데이트

STYLE_MAP = {
    3241660: "online",  # R.E.P.O
    2807960: "online",  # Battlefield™ 6
    730:     "online",  # Counter-Strike 2
    1808500: "online",  # ARC Raiders
    1030300: "story",   # Hollow Knight: Silksong
    570:     "online",  # Dota 2
    578080:  "online",  # PUBG
    2246340: "video",   # Monster Hunter Wilds
    2592160: "story",   # Dispatch
    553850:  "online",  # HELLDIVERS™ 2
    3240220: "online",  # Grand Theft Auto V Enhanced
    1091500: "story",   # Cyberpunk 2077
    1903340: "video",   # Clair Obscur: Expedition 33
    2001120: "story",   # Split Fiction
    1245620: "video",   # Elden Ring
    1086940: "video",   # Baldur's Gate 3
    1144200: "online",  # Ready or Not
    3167020: "video",   # Escape From Duckov
    3564740: "online",  # Where Winds Meet
    227300:  "video",   # Euro Truck Simulator 2
    108600:  "video",   # Project Zomboid
    413150:  "video",   # Stardew Valley
    1771300: "video",   # Kingdom Come 2
    3489700: "story",   # Stellar Blade™
    1172470: "online",  # Apex
    1222140: "story",   # Detroit: Become Human
    1326470: "video",   # Sons Of The Forest
    990080:  "story",   # Hogwarts Legacy
    1551360: "video",   # Forza Horizon 5
    1623730: "video",   # Palworld
    1145350: "video",   # Hades II
    2183900: "story",   # Space Marine AE
    230410:  "online",  # Warframe
    2139460: "online",  # Once Human
    236390:  "online",  # War Thunder
    440:     "online",  # Team Fortress 2
    1973530: "online",  # Limbus Company
    394360:  "video",   # Hearts of Iron IV
    3932890: "online",  # Escape from Tarkov
    526870:  "video",   # Satisfactory
    3513350: "online",  # Wuthering Waves
    3405690: "online",  # EA SPORTS FC™ 26
    2622380: "video",   # ELDEN RING NIGHTREIGN
    814380:  "video",   # Sekiro™: Shadows Die Twice - GOTY Edition
    648800:  "video",   # Raft
    3159330: "story",   # Assassin’s Creed Shadows
    3527290: "video",   # PEAK
    2651280: "story",   # Spider-Man 2
    294100:  "video",   # RimWorld
    1222670: "video",   # The Sims 4
}


# game_style 컬럼 생성
df_model["game_style"] = df_model["appid"].map(STYLE_MAP)
df_model["game_style"].value_counts()


game_style
online    2467704
video     1427383
story      935797
Name: count, dtype: int64

review 결측 appid별 분포 확인 구문

In [9]:
# appid별 review 결측치 분포
review_na_by_app = (
    df_model.groupby("appid")["review"]
      .apply(lambda s: s.isna().sum())
      .rename("review_na_cnt")
      .to_frame()
)

# appid별 전체 행 수
total_by_app = df_model.groupby("appid").size().rename("total_cnt").to_frame()

# 합치기 + 비율
review_na_stats = (
    total_by_app.join(review_na_by_app, how="left")
                .fillna({"review_na_cnt": 0})
)

review_na_stats["review_na_ratio"] = review_na_stats["review_na_cnt"] / review_na_stats["total_cnt"]

# 결측치 많은 순으로 확인
review_na_stats.sort_values("review_na_cnt", ascending=False).head(30)


,total_cnt,review_na_cnt,review_na_ratio
appid,,,
3241660,341851,1067,0.003121
2807960,301260,1051,0.003489
1808500,253341,991,0.003912
3240220,139431,675,0.004841
730,273327,668,0.002444
1091500,120404,566,0.004701
553850,148578,510,0.003433
578080,194464,500,0.002571
1030300,239171,493,0.002061


churn 라벨 생성 전 review가 NaN인 행 드롭

In [10]:
# review NaN 드롭
before = len(df_model)
df_model = df_model[df_model["review"].notna()].copy()
after = len(df_model)

print(f"[drop NaN review] before={before:,} -> after={after:,} (dropped {before-after:,})")


[drop NaN review] before=4,830,884 -> after=4,815,123 (dropped 15,761)


In [11]:
df_model[['review']].isna().sum()

review    0
dtype: int64

- appid별 공백/빈 문자열 리뷰 개수 확인 구문

In [12]:
# 공백/빈문자열(whitespace-only 포함) 마스크
blank_mask = df_model["review"].astype(str).str.strip().eq("")

blank_by_appid = (
    df_model.assign(is_blank_review=blank_mask)
            .groupby("appid")["is_blank_review"]
            .agg(total_cnt="size", blank_cnt="sum")
)

blank_by_appid["blank_ratio"] = blank_by_appid["blank_cnt"] / blank_by_appid["total_cnt"]

# 공백 리뷰가 있는 appid만, blank_cnt 큰 순으로 보기
blank_by_appid_nonzero = blank_by_appid[blank_by_appid["blank_cnt"] > 0].sort_values("blank_cnt", ascending=False)

display(blank_by_appid_nonzero.head(50))
print("공백 리뷰 총 개수:", int(blank_mask.sum()))

,total_cnt,blank_cnt,blank_ratio
appid,,,
730,272659,186,0.000682
3241660,340784,159,0.000467
3240220,138756,130,0.000937
2807960,300209,121,0.000403
578080,193964,111,0.000572
570,204123,94,0.000461
227300,92225,74,0.000802
1808500,252350,73,0.000289
1030300,238678,59,0.000247


공백 리뷰 총 개수: 1902


- appid별 공백/빈 문자열 리뷰 개수 확인 구문

공백/빈 문자열 리뷰 드롭

In [13]:
# 공백/빈 문자열 리뷰 드롭
before = len(df_model)

blank_mask = df_model["review"].astype(str).str.strip().eq("")
df_model = df_model[~blank_mask].copy()

after = len(df_model)
print(f"[drop blank review] before={before:,} -> after={after:,} (dropped {before-after:,})")


[drop blank review] before=4,815,123 -> after=4,813,221 (dropped 1,902)


churn 라벨 생성 (style별 기준일 적용)

In [14]:
# style별 churn 기준일(일 단위)
STYLE_WINDOW_DAYS = {
    "online": 7,
    "video": 10,
    "story": 5,
}

# 리뷰 시각 / 마지막 플레이 시각
review_dt = pd.to_datetime(df_model["timestamp_created"], unit="s", errors="coerce")
last_dt   = pd.to_datetime(df_model["last_played"], unit="s", errors="coerce")

# 리뷰 이후 며칠 뒤에 마지막 플레이가 있었는지
df_model["days_after_review"] = (last_dt - review_dt).dt.days

# game_style별 기준일 매핑 (none은 NaN)
df_model["churn_window_days"] = df_model["game_style"].map(STYLE_WINDOW_DAYS)

# 기본 churn: days_after_review < window 이면 churn=1 (떠난 것)
# - window가 없는(none) 행은 일단 NaN으로 둠(나중에 제외/처리)
df_model["churn"] = df_model["days_after_review"] < df_model["churn_window_days"].astype(int)


# 예외 처리(기존에 하던 규칙 유지)
df_model.loc[df_model["last_played"] == 0, "churn"] = 1
df_model.loc[df_model["days_after_review"] < 0, "churn"] = 1

print(df_model["churn"].value_counts(dropna=False).sort_index())


churn
False    3432753
True     1380468
Name: count, dtype: int64


C:\Users\user\AppData\Local\Temp\ipykernel_20300\831698950.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df_model.loc[df_model["last_played"] == 0, "churn"] = 1


In [15]:
df_model[['language']].nunique()

language    30
dtype: int64

In [16]:
df_model.describe()

,appid,recommendationid,steamid,num_games_owned,num_reviews_author,playtime_forever,playtime_last_two_weeks,playtime_at_review,last_played,timestamp_created,timestamp_updated,votes_up,votes_funny,weighted_vote_score,comment_count,days_after_review,churn_window_days
count,4.813221e+06,4.813221e+06,4.813221e+06,4.813221e+06,4.813221e+06,4.813221e+06,4.813221e+06,4.813221e+06,4.813221e+06,4.813221e+06,4.813221e+06,4.813221e+06,4.813221e+06,4.813221e+06,4.813221e+06,4.813221e+06,4.813221e+06
mean,1.637917e+06,2.034422e+08,7.656120e+16,7.224943e+01,1.093352e+01,1.398406e+04,4.346475e+02,1.067346e+04,1.761394e+09,1.756335e+09,1.756654e+09,1.038864e+00,1.820704e-01,5.029349e-01,6.856697e-02,5.806457e+01,7.498542e+00
std,1.156278e+06,8.342337e+06,6.190886e+08,2.492080e+02,6.318230e+01,4.066221e+04,1.151922e+03,3.810909e+04,1.145637e+07,8.849875e+06,8.803059e+06,3.084613e+01,6.968658e+00,2.391883e-02,2.379588e+00,1.368769e+02,1.784596e+00
min,4.400000e+02,1.848840e+08,7.656120e+16,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,5.000000e+00,0.000000e+00,1.736122e+09,1.736122e+09,0.000000e+00,0.000000e+00,2.213739e-02,0.000000e+00,-2.044100e+04,5.000000e+00
25%,5.780800e+05,1.967549e+08,7.656120e+16,0.000000e+00,2.000000e+00,1.500000e+03,0.000000e+00,6.490000e+02,1.759989e+09,1.749423e+09,1.749911e+09,0.000000e+00,0.000000e+00,5.000000e-01,0.000000e+00,4.000000e+00,7.000000e+00
50%,1.551360e+06,2.059368e+08,7.656120e+16,0.000000e+00,4.000000e+00,4.166000e+03,0.000000e+00,2.093000e+03,1.765025e+09,1.759660e+09,1.760116e+09,0.000000e+00,0.000000e+00,5.000000e-01,0.000000e+00,3.400000e+01,7.000000e+00
75%,2.807960e+06,2.102846e+08,7.656120e+16,6.900000e+01,1.000000e+01,1.034100e+04,2.330000e+02,6.120000e+03,1.767091e+09,1.764041e+09,1.764091e+09,0.000000e+00,0.000000e+00,5.000000e-01,0.000000e+00,1.020000e+02,1.000000e+01
max,3.932890e+06,2.152792e+08,7.656120e+16,3.816800e+04,1.974800e+04,2.919994e+06,3.664700e+04,2.789459e+06,1.767667e+09,1.767666e+09,1.767666e+09,2.009800e+04,6.372000e+03,9.958196e-01,4.500000e+03,3.640000e+02,1.000000e+01


- 텍스트 기반 good_review 파생변수 생성 구문

In [17]:
# 1) 언어별 키워드 사전
# - phrases: 문장/구문(부분일치 OK)
# - words: 단어성 키워드(라틴권은 단어경계 \b 적용)
# - neg: 부정 구문(걸리면 good=0으로 처리)
# - boundary: words에 \b를 붙일지 여부 (중국어/일본어/태국어/한국어는 보통 False)
LEXICON = {
    # English
    "english": {
        "phrases": [
            r"highly recommend(?:ed)?",
            r"definitely recommend",
            r"worth (?:buying|it|the money|the time)",
            r"great game",
            r"amazing game",
            r"awesome game",
            r"best game(?:s)?",
        ],
        "words": [
            r"awesome", r"amazing", r"great", r"excellent", r"fantastic", r"incredible",
            r"masterpiece", r"perfect", r"love", r"fun", r"enjoy", r"recommend", r"worth",
        ],
        "neg": [
            r"not\s+good", r"not\s+great", r"not\s+worth",
            r"(?:do\s*not|don't|dont)\s+recommend",
            r"(?:do\s*not|don't|dont)\s+buy",
            r"can't\s+recommend|cant\s+recommend",
            r"avoid\b", r"refund\b",
        ],
        "boundary": True,
    },

    # Spanish (Spain) + LatAm는 같이 처리
    "spanish": {
        "phrases": [r"muy bueno", r"vale la pena", r"lo recomiendo", r"recomendad[oa]"],
        "words": [r"genial", r"excelente", r"buen[oa]", r"incre[ií]ble", r"recomiendo", r"recomendar"],
        "neg": [r"no\s+recomiendo", r"no\s+vale\s+la\s+pena", r"no\s+es\s+buen[oa]", r"no\s+merece\s+la\s+pena", r"no\s+compr(?:es|ar)"],
        "boundary": True,
    },
    "latam": {  # 라틴아메리카 스페인어
        "phrases": [r"muy bueno", r"vale la pena", r"lo recomiendo", r"recomendad[oa]"],
        "words": [r"genial", r"excelente", r"buen[oa]", r"incre[ií]ble", r"recomiendo", r"recomendar"],
        "neg": [r"no\s+recomiendo", r"no\s+vale\s+la\s+pena", r"no\s+es\s+buen[oa]", r"no\s+merece\s+la\s+pena", r"no\s+compr(?:es|ar)"],
        "boundary": True,
    },

    # Portuguese (PT / BR)
    "portuguese": {
        "phrases": [r"vale a pena", r"recomendo", r"muito bom", r"jogo (?:muito )?bom"],
        "words": [r"ótimo", r"excelente", r"incr[ií]vel", r"perfeito", r"divertido", r"recomendar"],
        "neg": [r"não\s+recomendo", r"nao\s+recomendo", r"não\s+vale\s+a\s+pena", r"nao\s+vale\s+a\s+pena", r"não\s+é\s+bom", r"nao\s+e\s+bom", r"não\s+compr(?:e|ar)", r"nao\s+compr(?:e|ar)"],
        "boundary": True,
    },
    "brazilian": {  # 브라질 포르투갈어
        "phrases": [r"vale a pena", r"recomendo", r"muito bom", r"jogo (?:muito )?bom"],
        "words": [r"ótimo", r"excelente", r"incr[ií]vel", r"perfeito", r"divertido", r"recomendar"],
        "neg": [r"não\s+recomendo", r"nao\s+recomendo", r"não\s+vale\s+a\s+pena", r"nao\s+vale\s+a\s+pena", r"não\s+é\s+bom", r"nao\s+e\s+bom", r"não\s+compr(?:e|ar)", r"nao\s+compr(?:e|ar)"],
        "boundary": True,
    },

    # German
    "german": {
        "phrases": [r"sehr gut", r"klare(?:s)? empfehlung", r"lohnt sich", r"absolut empfehl"],
        "words": [r"genial", r"toll", r"super", r"großartig", r"exzellent", r"empfehle", r"empfehlenswert"],
        "neg": [r"nicht\s+empfehl", r"lohnt\s+sich\s+nicht", r"nicht\s+gut", r"kau(?:f|ft)\s+nicht", r"kein\s+kauf"],
        "boundary": True,
    },

    # French
    "french": {
        "phrases": [r"je recommande", r"vaut le coup", r"tr[eè]s bon", r"excellent jeu"],
        "words": [r"g[eé]nial", r"excellent", r"super", r"incroyable", r"parfait", r"recommande"],
        "neg": [r"je\s+ne\s+recommande\s+pas", r"ne\s+vaut\s+pas\s+le\s+coup", r"pas\s+bon", r"n['’]achetez\s+pas", r"n['’]ach[eè]te\s+pas"],
        "boundary": True,
    },

    # Italian
    "italian": {
        "phrases": [r"lo consiglio", r"vale la pena", r"molto bello", r"gioco (?:molto )?bello"],
        "words": [r"fantastico", r"ottimo", r"eccellente", r"stupendo", r"divertente", r"consiglio", r"consigliare"],
        "neg": [r"non\s+lo\s+consiglio", r"non\s+vale\s+la\s+pena", r"non\s+[eè]\s+bello", r"non\s+compr(?:are|atelo)"],
        "boundary": True,
    },

    # Dutch
    "dutch": {
        "phrases": [r"zeker aanraden", r"de moeite waard", r"heel goed", r"geweldig spel"],
        "words": [r"geweldig", r"fantastisch", r"super", r"leuk", r"aanraden", r"aanbevelen", r"waarde"],
        "neg": [r"niet\s+aanrad", r"niet\s+de\s+moeite\s+waard", r"niet\s+goed", r"koop\s+niet"],
        "boundary": True,
    },

    # Swedish / Norwegian / Danish / Finnish
    "swedish": {
        "phrases": [r"rekommenderar", r"värt det", r"jättebra", r"riktigt bra"],
        "words": [r"fantastisk", r"grym", r"suverän", r"toppen", r"kul", r"rekommendera", r"värd"],
        "neg": [r"rekommenderar\s+inte", r"inte\s+värt", r"inte\s+bra", r"köp\s+inte"],
        "boundary": True,
    },
    "norwegian": {
        "phrases": [r"anbefaler", r"verdt det", r"kjempebra", r"veldig bra"],
        "words": [r"fantastisk", r"råbra", r"suveren", r"gøy", r"anbefale", r"verdt"],
        "neg": [r"anbefaler\s+ikke", r"ikke\s+verdt", r"ikke\s+bra", r"ikke\s+kjøp"],
        "boundary": True,
    },
    "danish": {
        "phrases": [r"anbefaler", r"v[æa]rd at", r"mega god", r"rigtig god"],
        "words": [r"fantastisk", r"fremragende", r"super", r"sjov", r"anbefale", r"v[æa]rd"],
        "neg": [r"anbefaler\s+ikke", r"ikke\s+v[æa]rd", r"ikke\s+god", r"k[oø]b\s+ikke"],
        "boundary": True,
    },
    "finnish": {
        "phrases": [r"suosittelen", r"todella hyv[äa]", r"sen arvoinen", r"hyv[äa] peli"],
        "words": [r"loistava", r"mahtava", r"erinomainen", r"hauska", r"suositella", r"arvoinen"],
        "neg": [r"en\s+suosittele", r"ei\s+kannata", r"ei\s+hyv[äa]", r"älä\s+osta"],
        "boundary": True,
    },

    # Polish / Czech / Romanian / Hungarian / Bulgarian / Greek / Ukrainian / Russian / Turkish
    "polish": {
        "phrases": [r"polecam", r"warto", r"świetna gra", r"bardzo dobra"],
        "words": [r"świetn[aey]", r"super", r"rewelacyjna", r"doskonała", r"polecić", r"warto"],
        "neg": [r"nie\s+polecam", r"nie\s+warto", r"nie\s+jest\s+dobr", r"nie\s+kupuj"],
        "boundary": True,
    },
    "czech": {
        "phrases": [r"doporu[čc]uji", r"stoj[ií]\s+za\s+to", r"skv[ěe]l[aá]", r"v[ýy]born[aá]"],
        "words": [r"super", r"skv[ěe]l", r"v[ýy]born", r"bav[ií]", r"doporu[čc]it"],
        "neg": [r"nedoporu[čc]uji", r"nestoj[ií]\s+za\s+to", r"nen[ií]\s+dobr", r"nekupuj"],
        "boundary": True,
    },
    "romanian": {
        "phrases": [r"recomand", r"merit[ăa]", r"foarte bun", r"joc (?:foarte )?bun"],
        "words": [r"excelent", r"minunat", r"super", r"recomanda", r"merit"],
        "neg": [r"nu\s+recomand", r"nu\s+merit[ăa]", r"nu\s+e\s+bun", r"nu\s+cump[ăa]ra"],
        "boundary": True,
    },
    "hungarian": {
        "phrases": [r"aj[aá]nlom", r"meg[eé]ri", r"nagyon j[oó]", r"szuper j[aá]t[eé]k"],
        "words": [r"szuper", r"fantasztikus", r"kiv[aá]l[oó]", r"nagyon", r"aj[aá]nlani", r"meg[eé]r"],
        "neg": [r"nem\s+aj[aá]nlom", r"nem\s+[eé]ri\s+meg", r"nem\s+j[oó]", r"ne\s+vedd\s+meg"],
        "boundary": True,
    },
    "bulgarian": {
        "phrases": [r"препоръч", r"много добра", r"страхотна", r"заслужава си"],
        "words": [r"страхот", r"отлич", r"супер", r"препоръч", r"шедьовър"],
        "neg": [r"не\s+препоръч", r"не\s+си\s+струва", r"не\s+е\s+доб", r"не\s+купувай"],
        "boundary": False,  # кир릴은 \b가 애매해서 단순부분일치로
    },
    "greek": {
        "phrases": [r"το\s+προτείν", r"αξίζει", r"πολύ\s+καλ", r"εξαιρετικ"],
        "words": [r"τέλει", r"φοβε", r"εξαιρετικ", r"καταπληκτικ", r"προτείν", r"αξίζ"],
        "neg": [r"δεν\s+προτείν", r"δεν\s+αξίζ", r"δεν\s+είναι\s+καλ", r"μην\s+αγοράσ"],
        "boundary": False,
    },
    "ukrainian": {
        "phrases": [r"рекоменд", r"дуже\s+хорош", r"варто", r"чудов"],
        "words": [r"відмін", r"класн", r"шедевр", r"рекоменд", r"варто"],
        "neg": [r"не\s+рекоменд", r"не\s+варто", r"не\s+хорош", r"не\s+купуй"],
        "boundary": False,
    },
    "russian": {
        "phrases": [r"рекоменд", r"очень\s+хорош", r"стоит", r"шедевр"],
        "words": [r"отлич", r"классн", r"супер", r"шедевр", r"рекоменд", r"стоит"],
        "neg": [r"не\s+рекоменд", r"не\s+стоит", r"плох", r"не\s+покупай", r"не\s+берите"],
        "boundary": False,
    },
    "turkish": {
        "phrases": [r"kesinlikle tavsiye", r"tavsiye ederim", r"çok iyi", r"mükemmel", r"harika"],
        "words": [r"güzel", r"mükemmel", r"harika", r"şahane", r"tavsiye", r"değer"],
        "neg": [r"tavsiye etmem", r"tavsiye etmiyorum", r"iyi değil", r"alma", r"almayın", r"değmez"],
        "boundary": True,
    },

    # Korean / Japanese / Chinese / Arabic / Thai / Vietnamese / Indonesian
    "koreana": {
        "phrases": [r"강추", r"완전 추천", r"강력 추천", r"갓겜", r"명작", r"존잼", r"개꿀잼", r"재밌", r"재미있"],
        "words": [r"추천", r"최고", r"꿀잼", r"재미", r"좋다", r"훌륭", r"완벽", r"감동"],
        "neg": [r"비추", r"추천\s*안", r"추천\s*하지", r"재미없", r"별로", r"최악", r"사지\s*마", r"사지마", r"환불"],
        "boundary": False,
    },
    "japanese": {
        "phrases": [r"おすすめ", r"オススメ", r"最高", r"神ゲー", r"買う価値", r"面白い", r"楽しい"],
        "words": [r"おすすめ", r"最高", r"神", r"面白", r"楽しい", r"良い", r"素晴らしい"],
        "neg": [r"おすすめしない", r"買わない方が", r"つまらない", r"面白くない", r"最悪", r"返品"],
        "boundary": False,
    },
    "schinese": {
        "phrases": [r"强烈推荐", r"非常推荐", r"值得买", r"值得入", r"很值得", r"很好玩", r"神作", r"精品"],
        "words": [r"推荐", r"值得", r"好玩", r"很好", r"优秀", r"完美", r"喜欢"],
        "neg": [r"不推荐", r"不值得", r"不好玩", r"垃圾", r"别买", r"千万别买", r"退款"],
        "boundary": False,
    },
    "tchinese": {
        "phrases": [r"強烈推薦", r"非常推薦", r"值得買", r"值得入", r"很值得", r"很好玩", r"神作", r"精品"],
        "words": [r"推薦", r"值得", r"好玩", r"很好", r"優秀", r"完美", r"喜歡"],
        "neg": [r"不推薦", r"不值得", r"不好玩", r"垃圾", r"別買", r"千萬別買", r"退款"],
        "boundary": False,
    },
    "arabic": {
        "phrases": [r"أنصح", r"ممتاز", r"رائع", r"يستحق", r"لعبة رائعة", r"ممتعة"],
        "words": [r"ممتاز", r"رائع", r"جميل", r"ممتع", r"يستحق", r"أنصح"],
        "neg": [r"لا\s+أنصح", r"لا\s+يستحق", r"سيئ", r"لا\s+تشتري", r"استرجاع"],
        "boundary": False,
    },
    "thai": {
        "phrases": [r"แนะนำ", r"ดีมาก", r"สุดยอด", r"คุ้มค่า", r"สนุกมาก", r"โคตรสนุก"],
        "words": [r"แนะนำ", r"ดี", r"สนุก", r"สุดยอด", r"คุ้ม", r"ชอบ"],
        "neg": [r"ไม่แนะนำ", r"ไม่คุ้ม", r"ไม่ดี", r"แย่", r"อย่าซื้อ", r"ขอคืนเงิน"],
        "boundary": False,
    },
    "vietnamese": {
        "phrases": [r"rất hay", r"tuyệt vời", r"đáng mua", r"đáng tiền", r"nên mua", r"khuyên dùng"],
        "words": [r"hay", r"tuyệt", r"xuất sắc", r"đáng", r"thích", r"khuyên", r"nên"],
        "neg": [r"không\s+khuyên", r"không\s+đáng", r"đừng\s+mua", r"tệ", r"chán", r"hoàn tiền"],
        "boundary": True,
    },
    "indonesian": {
        "phrases": [r"sangat bagus", r"rekomendasi", r"worth it", r"layak dibeli", r"seru banget"],
        "words": [r"bagus", r"keren", r"mantap", r"seru", r"rekomend", r"layak"],
        "neg": [r"tidak\s+rekomend", r"jangan\s+beli", r"tidak\s+layak", r"jelek", r"buruk", r"refund"],
        "boundary": True,
    },
}

# 없는 언어는 english로 fallback
DEFAULT_LANG = "english"


# 2) 정규식 빌더
def _compile_lexicon(cfg):
    boundary = cfg.get("boundary", True)

    parts_good = []
    for p in cfg.get("phrases", []):
        parts_good.append(f"(?:{p})")
    for w in cfg.get("words", []):
        if boundary:
            parts_good.append(rf"\b{w}\b")
        else:
            parts_good.append(f"(?:{w})")

    good_pat = "|".join(parts_good) if parts_good else r"$^"
    good_re = re.compile(good_pat, flags=re.UNICODE)

    neg_parts = [f"(?:{p})" for p in cfg.get("neg", [])]
    neg_pat = "|".join(neg_parts) if neg_parts else r"$^"
    neg_re = re.compile(neg_pat, flags=re.UNICODE)

    return good_re, neg_re


_COMPILED = {}
for lang, cfg in LEXICON.items():
    _COMPILED[lang] = _compile_lexicon(cfg)
_COMPILED[DEFAULT_LANG] = _COMPILED.get(DEFAULT_LANG, _compile_lexicon(LEXICON["english"]))


def add_good_flag_multilang(df_model, text_col="review", lang_col="language"):
    out = df_model.copy()

    text = out[text_col].fillna("").astype(str).str.casefold()
    lang = out[lang_col].fillna(DEFAULT_LANG).astype(str)

    good_hit = pd.Series(False, index=out.index)
    neg_hit  = pd.Series(False, index=out.index)

    for l in lang.unique():
        mask = (lang == l)
        good_re, neg_re = _COMPILED.get(l, _COMPILED[DEFAULT_LANG])

        good_hit.loc[mask] = text.loc[mask].str.contains(good_re, regex=True)
        neg_hit.loc[mask]  = text.loc[mask].str.contains(neg_re,  regex=True)

    out["good_review"] = (good_hit & (~neg_hit)).astype(int)
    return out


# df_good = add_good_flag_multilang(df, text_col="review", lang_col="language")
# print(df_good["good_review"].value_counts())
df_model = add_good_flag_multilang(df_model, text_col="review", lang_col="language")
print(df_model["good_review"].value_counts())

good_review
0    3525940
1    1287281
Name: count, dtype: int64


- 새로운 파생변수 생성 구문

In [18]:
### 추가한 파생변수..

# 1. 리뷰 시점 플레이 집중도 (플레이타임 “규모” → “강도 / 몰입도”로 바꾸기)
# 게임 많이 가진데 이 게임만 오래 함 → 비이탈 / 게임 많고 얕게만 함 → 이탈 가능성↑
df_model['playtime_per_game'] = np.log1p(df_model['playtime_at_review'] / (df_model['num_games_owned'] + 1))


# 2. 리뷰 작성 시점의 몰입 단계 => churn과 매우 잘 갈림 (충성도?)

# log 변환
df_model['log_playtime'] = np.log1p(df_model['playtime_at_review'])
# 구간 나누기
bins = [-np.inf, 4, 8, np.inf]
labels = ['short', 'mid', 'long']

df_model['playtime_stage'] = pd.cut(
    df_model['log_playtime'],
    bins=bins,
    labels=labels
)
# 원하는 이름으로 0/1 컬럼 만들기
df_model['is_short_play'] = (df_model['playtime_stage'] == 'short').astype(int)
df_model['is_mid_play']   = (df_model['playtime_stage'] == 'mid').astype(int)
df_model['is_long_play']  = (df_model['playtime_stage'] == 'long').astype(int)



# *** 유저 성향 피처 (엄청 중요!)
# 3. 리뷰어 성향 (리뷰 중독자 vs 일반 유저)
# 리뷰를 많이 쓰는 사람 → 서비스 잔존율 높음 / 리뷰 거의 안 쓰는 사람 → 이탈 가능성↑ (충성도?)
df_model['reviews_per_game'] = np.log1p(df_model['num_reviews_author'] / (df_model['num_games_owned'] + 1))



# 4. “경험 많은/적은 유저” 여부
df_model['log_num_games_owned'] = np.log1p(df_model['num_games_owned'])
df_model['log_num_reviews_author'] = np.log1p(df_model['num_reviews_author'])

# 헤비 유저
df_model['is_heavy_user'] = (
    (df_model['log_num_games_owned'] > 5.0) & 
    (df_model['log_num_reviews_author'] > 3.5)
).astype(int)
# 라이트 유저 (헤비보다 더 중요한 컬럼.)
df_model['is_light_user'] = (
    (df_model['log_num_games_owned'] < 2.0) &
    (df_model['log_num_reviews_author'] < 1.0)
).astype(int)

# 5. 리뷰 감정 × 행동 결합 피처 (***핵심!!!)
# 5-1. 긍정 리뷰 + 낮은 플레이타임 = 위험
df_model['positive_but_short_play'] = (
    (df_model['good_review'] == 1) & 
    (df_model['playtime_at_review'] < 60)
).astype(int)
# “좋다고는 했는데 금방 떠난 사람” → churn에 엄청 강함

# 5-2. 부정 리뷰 + 높은 플레이타임
df_model['negative_but_long_play'] = (
    (df_model['good_review'] == 0) & 
    (df_model['playtime_at_review'] > 1200)
).astype(int)
# -> 욕하면서 계속 하는 타입 = 비이탈



# 6. 사회적 반응 파생 피처 (반응 여부 자체)
df_model['has_votes'] = ((df_model['votes_up'] + df_model['votes_funny']) > 0).astype(int)
df_model['has_comment'] = (df_model['comment_count'] > 0).astype(int)



# +
# 7. 리뷰 업데이트의 진정성 (Review Freshness) (아주 좋다고 합니다!!)
# timestamp_created와 timestamp_updated가 다른 유저는 게임에 아주 깊게 관여하고 있는 유저이다.
# 리뷰를 수정했다는 것은 이탈하지 않고 게임을 지속하며 의견을 업데이트했다는 강력한 증거.
# (비이탈 그룹에서 높게 나타날 확률이 큼)
# # 리뷰 수정 여부 (관여도)
df_model['is_updated_review'] = (df_model['timestamp_created'] != df_model['timestamp_updated']).astype(int)

# +
# 8. 커뮤니티 신뢰도 밀도 (Social Density)
# votes_up이나 comment_count는 데이터가 매우 희소(sparse).
# 이를 개별로 두지 않고, 유저의 '활동량' 대비 얼마나 많은 반응을 끌어냈는지 비율로 전환하면 모델이 학습하기 훨씬 수월해진다.
# 평소 리뷰를 100개 쓰는 사람이 추천 1개 받는 것보다, 리뷰를 1개 썼는데 추천 1개를 받는 것이 훨씬 '핵심 유저'일 가능성이 높다.
# 영향력 있는 리뷰어는 커뮤니티 활동 때문에라도 이탈률이 낮게 측정되는 경우가 많다.
# # 리뷰 1개당 평균 반응도 (커뮤니티 영향력)
df_model['social_density'] = np.log1p((df_model['votes_up'] + df_model['comment_count']) / (df_model['num_reviews_author'] + 1))
# 이거는 'has_votes', 'has_comment' 조금 겹칠 수 도 있는데 굳이 안빼도 된다고 합니다.
# 제거할 거면 모델 학습 완료하고, feature importance / coefficient 확인 헤서 위 두개와 비교 후에 제거 하는게 좋다고 합니다!


# 안전 처리 (inf, -inf 제거)
df_model.replace([np.inf, -np.inf], 0, inplace=True)

df_model.head(3)

,appid,recommendationid,steamid,num_games_owned,num_reviews_author,playtime_forever,playtime_last_two_weeks,playtime_at_review,last_played,language,...,log_num_games_owned,log_num_reviews_author,is_heavy_user,is_light_user,positive_but_short_play,negative_but_long_play,has_votes,has_comment,is_updated_review,social_density
27798,2139460,215256415,76561198092089560,0,1,1659,1659,1628.0,1767647101,english,...,0.0,0.693147,0,1,0,1,0,0,0,0.0
27799,2139460,215256182,76561197995642012,0,4,370,367,339.0,1767646994,french,...,0.0,1.609438,0,0,0,0,0,0,0,0.0
27800,2139460,215249671,76561198217416651,0,32,552,552,401.0,1767648127,greek,...,0.0,3.496508,0,0,0,0,0,0,0,0.0


In [19]:
# df_model[['language', 'voted_up', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'primarily_steam_deck', 'game_style']]

# 'language' ('english', 'koreana', ...), 'game_style' ('online','video','story') 인코딩 (True/False)
# df_model['language'].value_counts()     # 카테고리가 많아 상위 10개 미만 값들은 others로 통합.

# 상위 10개 언어 추출
top_n = 10
top_langs = df_model['language'].value_counts().head(top_n).index

# 상위 10개만 유지, 나머지는 other
df_model['language'] = df_model['language'].where(
    df_model['language'].isin(top_langs),
    'other'
)

df_model = pd.get_dummies(                  #get_dummies 는 columns 를 자동 제거
    df_model,
    columns=['language', 'game_style'],
    drop_first=True
)

In [20]:
# 생성된 컬럼 확인
print([col for col in df_model.columns if col.startswith('language_')])
print([col for col in df_model.columns if col.startswith('game_style_')]) # game_style_online은 둘다 ('game_style_story', 'game_style_video') 0 인 것

['language_english', 'language_french', 'language_german', 'language_koreana', 'language_other', 'language_polish', 'language_russian', 'language_schinese', 'language_spanish', 'language_turkish']
['game_style_story', 'game_style_video']


In [21]:
# 원핫인코딩 (True 1, False 0)

# df_model[['voted_up', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'primarily_steam_deck']]
# 인코딩된 언어 칼럼들: ['language_english', 'language_french', 'language_german','language_koreana','language_other','language_polish','language_russian','language_schinese','language_spanish','language_turkish']
# 인코딩된 게임 스타일 칼럼들: ['game_style_story', 'game_style_video']

bool_cols = ['voted_up', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'primarily_steam_deck',
            'language_english', 'language_french', 'language_german','language_koreana','language_other','language_polish','language_russian','language_schinese','language_spanish','language_turkish',
            'game_style_story', 'game_style_video'
            ]

for col in bool_cols:
    df_model[col] = df_model[col].astype(int)

# df_model['voted_up'].value_counts()
# df_model['language_english'].value_counts()
# df_model['game_style_story'].value_counts()

- 학습용 피처 목록 확정 + top50 appid 잡기

In [22]:
TARGET = "churn"

#  최종 확정한 피처
FEATURES = [
    "voted_up",
    "steam_purchase",
    "received_for_free",
    "written_during_early_access",
    "primarily_steam_deck",

    # 원핫된 language/game_style 컬럼들 (너가 만든 컬럼명 그대로)
    "language_english", "language_french", "language_german", "language_koreana",
    "language_other", "language_polish", "language_russian", "language_schinese",
    "language_spanish", "language_turkish",
    "game_style_story", "game_style_video",

    # 파생 피처들
    "weighted_vote_score",
    "playtime_per_game",
    "is_short_play", "is_mid_play", "is_long_play",
    "reviews_per_game",
    "is_heavy_user",
    "positive_but_short_play",
    "negative_but_long_play",
    "has_votes",
    "has_comment",
    "social_density",
    "is_updated_review",
]

# 혹시 누락/오타로 없는 컬럼이 있으면 자동 제외(코드 안깨지게)
FEATURES = [c for c in FEATURES if c in df_model.columns]

# top50 appid (리뷰 많은 순)
top_appids = (
    df_model.groupby("appid")
            .size()
            .sort_values(ascending=False)
            .head(50)
            .index.tolist()
)

print("n_features:", len(FEATURES))
print("top_appids(50) sample:", top_appids[:5])

n_features: 30
top_appids(50) sample: [3241660, 2807960, 730, 1808500, 1030300]


- 게임별 학습 루프 (로지스틱=스케일 / 트리=노스케일) + 결과 테이블

In [23]:
RANDOM_STATE = 42
TEST_SIZE = 0.2

# 모델: 로지스틱(스케일링) / 트리2개(스케일링 없음)
models = {
    "Logistic": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced"
    ),
    "HistGB": HistGradientBoostingClassifier(random_state=RANDOM_STATE)
}

def eval_metrics(y_true, y_pred):
    return {
        "acc": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "pos_rate": float(np.mean(y_pred == 1))
    }

results = []
result_list = []

for appid in top_appids:
    gdf = df_model[df_model["appid"] == appid]

    # 타깃이 한쪽만 있으면 학습 불가
    if gdf[TARGET].nunique() < 2:
        continue

    X = gdf[FEATURES]
    y = gdf[TARGET].astype(int)

    # stratify split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )

    for name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        m = eval_metrics(y_test, pred)
        results.append({
            "appid": int(appid),
            "n_rows": int(len(gdf)),
            "churn_rate": float(y.mean()),
            "model": name,
            **m
        })
        result_list.append({
            "appid": int(appid),
            "n_rows": int(len(gdf)),
            "churn_rate": float(y.mean()),
            "model": name,
            **m,
            "model_obj":model
        })

    # result_list 에서 점수가 가장 높은 하나를 선정

    # result_list_df = pd.DataFrame(result_list).sort_values(["f1", "recall"], ascending=False).head(1)

    # appid = result_list_df["appid"]
    # model_obj = result_list_df["model_obj"]
    
    # path = f"model/model_{appid}.pkl"  # 동적 파일명

    # joblib.dump(model_obj, path)
    
    # # result_list 초기화
    # result_list = []

results_df = pd.DataFrame(results).sort_values(["f1", "recall"], ascending=False).reset_index(drop=True)

print("done. rows:", len(results_df))
display(results_df.head(30))



done. rows: 150


,appid,n_rows,churn_rate,model,acc,precision,recall,f1,pos_rate
0,3527290,31721,0.699001,HistGB,0.703388,0.715589,0.955355,0.818270,0.933176
1,1222140,64366,0.648681,HistGB,0.656750,0.661306,0.965154,0.784848,0.946714
2,2592160,150629,0.594527,HistGB,0.689836,0.695889,0.849590,0.765096,0.725851
3,3527290,31721,0.699001,RandomForest,0.645705,0.719799,0.807441,0.761105,0.784082
4,3527290,31721,0.699001,Logistic,0.644287,0.764577,0.709583,0.736054,0.648700
5,2001120,109621,0.562949,HistGB,0.659932,0.677284,0.756299,0.714614,0.628643
6,1222140,64366,0.648681,RandomForest,0.594143,0.665572,0.752365,0.706312,0.733261
7,2592160,150629,0.594527,Logistic,0.609772,0.643080,0.772263,0.701776,0.713968
8,2592160,150629,0.594527,RandomForest,0.629290,0.677719,0.717827,0.697196,0.629722
9,1903340,119542,0.546661,HistGB,0.598185,0.610505,0.731905,0.665716,0.655360


### Lightgmb 추가 구문

In [24]:
from tqdm.auto import tqdm
import os
from lightgbm import LGBMClassifier
import joblib

os.makedirs("model", exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

models = {
    "Logistic": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
    ]),
    "LightGBM": LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced"
    ),
    "HistGB": HistGradientBoostingClassifier(random_state=RANDOM_STATE)
}

def eval_metrics(y_true, y_pred):
    return {
        "acc": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "pos_rate": float(np.mean(y_pred == 1))
    }

results = []
result_list = []

# 바깥 appid 진행바
for appid in tqdm(top_appids, desc="Games", unit="game"):
    gdf = df_model[df_model["appid"] == appid]

    if gdf[TARGET].nunique() < 2:
        continue

    X = gdf[FEATURES]
    y = gdf[TARGET].astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )

    # 안쪽 모델 진행바 (leave=False로 깔끔하게)
    for name, model in tqdm(models.items(), desc=f"Models (appid={appid})", leave=False):
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        m = eval_metrics(y_test, pred)
        results.append({
            "appid": int(appid),
            "n_rows": int(len(gdf)),
            "churn_rate": float(y.mean()),
            "model": name,
            **m
        })
        result_list.append({
            "appid": int(appid),
            "n_rows": int(len(gdf)),
            "churn_rate": float(y.mean()),
            "model": name,
            **m,
            "model_obj": model
        })

    # appid별 최고 모델 1개 저장
    result_list_df = (
        pd.DataFrame(result_list)
          .sort_values(["f1", "recall"], ascending=False)
          .head(1)
    )

    best_row = result_list_df.iloc[0]
    best_appid = int(best_row["appid"])
    best_model = best_row["model_obj"]

    path = f"model/model_{best_appid}.pkl"
    joblib.dump(best_model, path)

    # 다음 appid를 위해 초기화
    result_list = []

results_df = (
    pd.DataFrame(results)
      .sort_values(["f1", "recall"], ascending=False)
      .reset_index(drop=True)
)

print("done. rows:", len(results_df))
display(results_df.head(50))


c:\Users\user\skn_2nd_team1\skn2_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Games:   0%|          | 0/50 [00:00<?, ?game/s]

[LightGBM] [Info] Number of positive: 59923, number of negative: 212577
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.030133 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1066
[LightGBM] [Info] Number of data points in the train set: 272500, number of used features: 27
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


Games:   2%|▏         | 1/50 [00:18<15:16, 18.70s/game]

[LightGBM] [Info] Number of positive: 37878, number of negative: 202192
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.025732 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1064
[LightGBM] [Info] Number of data points in the train set: 240070, number of used features: 26
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


Games:   4%|▍         | 2/50 [00:34<13:49, 17.28s/game]

...

Games:  98%|█████████▊| 49/50 [06:27<00:04,  4.90s/game]

[LightGBM] [Info] Number of positive: 8525, number of negative: 15917
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008938 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1059
[LightGBM] [Info] Number of data points in the train set: 24442, number of used features: 27
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


Games: 100%|██████████| 50/50 [06:32<00:00,  7.85s/game]

done. rows: 150


,appid,n_rows,churn_rate,model,acc,precision,recall,f1,pos_rate
0,3527290,31721,0.699001,HistGB,0.703388,0.715589,0.955355,0.818270,0.933176
1,1222140,64366,0.648681,HistGB,0.656750,0.661306,0.965154,0.784848,0.946714
2,2592160,150629,0.594527,HistGB,0.689836,0.695889,0.849590,0.765096,0.725851
3,2592160,150629,0.594527,LightGBM,0.685023,0.709440,0.796382,0.750401,0.667397
4,3527290,31721,0.699001,Logistic,0.644287,0.764577,0.709583,0.736054,0.648700
5,3527290,31721,0.699001,LightGBM,0.629157,0.763811,0.679594,0.719246,0.621907
6,2001120,109621,0.562949,HistGB,0.659932,0.677284,0.756299,0.714614,0.628643
7,2592160,150629,0.594527,Logistic,0.609772,0.643080,0.772263,0.701776,0.713968
8,2001120,109621,0.562949,LightGBM,0.651403,0.703182,0.658916,0.680330,0.527526
9,1903340,119542,0.546661,HistGB,0.598185,0.610505,0.731905,0.665716,0.655360


- 게임별 최고 모델 출력

In [25]:
best_by_game = (
    results_df.sort_values(["appid", "f1", "recall"], ascending=[True, False, False])
              .groupby("appid", as_index=False)
              .head(1)
              .sort_values("f1", ascending=False)
              .reset_index(drop=True)
)

display(best_by_game.head(50))


,appid,n_rows,churn_rate,model,acc,precision,recall,f1,pos_rate
0,3527290,31721,0.699001,HistGB,0.703388,0.715589,0.955355,0.818270,0.933176
1,1222140,64366,0.648681,HistGB,0.656750,0.661306,0.965154,0.784848,0.946714
2,2592160,150629,0.594527,HistGB,0.689836,0.695889,0.849590,0.765096,0.725851
3,2001120,109621,0.562949,HistGB,0.659932,0.677284,0.756299,0.714614,0.628643
4,1903340,119542,0.546661,HistGB,0.598185,0.610505,0.731905,0.665716,0.655360
5,1145350,51398,0.410814,Logistic,0.623249,0.526709,0.817192,0.640557,0.637354
6,3564740,93627,0.323475,LightGBM,0.732030,0.575410,0.654449,0.612390,0.367884
7,3167020,96654,0.487222,LightGBM,0.637887,0.651656,0.551603,0.597470,0.412395
8,3932890,41232,0.303817,LightGBM,0.711531,0.519213,0.684757,0.590604,0.400752
9,570,204029,0.193169,LightGBM,0.809342,0.504841,0.674702,0.577541,0.258148


In [26]:
best_df = best_by_game.copy() 

F1_LOW = 0.50
POS_GAP = 0.25
POS_MULT = 1.8

best_df["pos_gap"] = best_df["pos_rate"] - best_df["churn_rate"]
best_df["pos_mult"] = best_df["pos_rate"] / (best_df["churn_rate"] + 1e-12)

need_tune = best_df[
    (best_df["f1"] < F1_LOW) |
    (best_df["pos_gap"] > POS_GAP) |
    ((best_df["churn_rate"] < 0.35) & (best_df["pos_mult"] > POS_MULT))
].copy()

MAX_TUNE_GAMES = 12
need_tune = need_tune.sort_values("f1", ascending=True).head(MAX_TUNE_GAMES)

tune_appids = need_tune["appid"].astype(int).tolist()

print("튜닝 대상 appid 개수:", len(tune_appids))
display(need_tune[["appid","n_rows","churn_rate","model","acc","precision","recall","f1","pos_rate","pos_gap","pos_mult"]])

튜닝 대상 appid 개수: 12


,appid,n_rows,churn_rate,model,acc,precision,recall,f1,pos_rate,pos_gap,pos_mult
49,1973530,43061,0.084206,LightGBM,0.812609,0.242617,0.577931,0.341762,0.200511,0.116305,2.381191
48,730,272473,0.184881,LightGBM,0.616937,0.253447,0.550968,0.347187,0.401908,0.217028,2.173880
47,553850,148012,0.139239,LightGBM,0.703915,0.251685,0.570839,0.349343,0.315813,0.176574,2.268138
46,3405690,36162,0.174050,LightGBM,0.674962,0.271930,0.517077,0.356419,0.330983,0.156933,1.901653
45,1551360,53036,0.217230,Logistic,0.632259,0.299724,0.518663,0.379908,0.375848,0.158619,1.730188
44,1808500,252277,0.186660,LightGBM,0.695121,0.307344,0.505203,0.382184,0.306822,0.120162,1.643748
43,3240220,138626,0.219338,LightGBM,0.646397,0.310881,0.503207,0.384326,0.355010,0.135671,1.618548
42,227300,92151,0.227171,Logistic,0.606098,0.301357,0.556723,0.391042,0.419673,0.192503,1.847393
41,294100,30831,0.233110,Logistic,0.597373,0.303498,0.561196,0.393947,0.431166,0.198056,1.849628
40,1245620,108874,0.265821,Logistic,0.613456,0.345915,0.509848,0.412180,0.391780,0.125959,1.473847


In [27]:
from tqdm import tqdm
import joblib

RANDOM_STATE = 42
TEST_SIZE = 0.20
VAL_SIZE_IN_TRAIN = 0.20

THR_GRID = np.round(np.arange(0.10, 0.91, 0.02), 2)

CV_SPLITS = 2
N_JOBS_GRID = 4   # 느리면 2로

TARGET = "churn"

print("FEATURES 개수:", len(FEATURES))

def get_proba(clf, X):
    return clf.predict_proba(X)[:, 1]

def metrics(y_true, y_pred):
    return {
        "acc": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "pos_rate": float(np.mean(y_pred == 1))
    }

def best_threshold_by_f1(y_true, proba, thr_grid=THR_GRID):
    best_f1, best_thr = -1.0, 0.5
    for thr in thr_grid:
        pred = (proba >= thr).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, float(thr)
    return best_thr

candidates = [
    ("Logistic",
     Pipeline([("scaler", StandardScaler()),
               ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))]),
     {"clf__C": [0.5, 1.0, 2.0]}),

    ("LightGBM",
     Pipeline([("clf", LGBMClassifier(
         random_state=RANDOM_STATE,
         n_jobs=-1,
         class_weight="balanced"
     ))]),
     {
         "clf__n_estimators": [300, 600],
         "clf__learning_rate": [0.05, 0.1],
         "clf__num_leaves": [31, 63],
         "clf__subsample": [0.8, 1.0],
         "clf__colsample_bytree": [0.8, 1.0],
     }),

    ("HistGB",
     Pipeline([("clf", HistGradientBoostingClassifier(random_state=RANDOM_STATE))]),
     {"clf__learning_rate": [0.05, 0.1],
      "clf__max_leaf_nodes": [31, 63],
      "clf__min_samples_leaf": [20, 50]})
]

tuned_rows = []
tuned_list = []
cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for appid in tqdm(tune_appids, desc="Tuning games"):
    gdf = df_model[df_model["appid"] == appid].copy()

    # 방어
    gdf = gdf.replace([np.inf, -np.inf], 0)
    if gdf[TARGET].nunique() < 2:
        continue

    X = gdf[FEATURES]
    y = gdf[TARGET].astype(int)

    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=VAL_SIZE_IN_TRAIN, random_state=RANDOM_STATE, stratify=y_train_full
    )

    best_game = None

    for name, pipe, grid in candidates:
        gs = GridSearchCV(
            pipe, grid, scoring="f1", cv=cv, n_jobs=N_JOBS_GRID, refit=True
        )
        gs.fit(X_train, y_train)

        best_est = gs.best_estimator_

        # val에서 threshold 최적화
        val_proba = get_proba(best_est, X_val)
        best_thr = best_threshold_by_f1(y_val, val_proba)

        # train/test 평가
        tr_proba = get_proba(best_est, X_train_full)
        te_proba = get_proba(best_est, X_test)

        tr_pred = (tr_proba >= best_thr).astype(int)
        te_pred = (te_proba >= best_thr).astype(int)

        tr_m = metrics(y_train_full, tr_pred)
        te_m = metrics(y_test, te_pred)

        row = {
            "appid": int(appid),
            "n_rows": int(len(gdf)),
            "churn_rate": float(y.mean()),
            "model": name,
            "best_thr": float(best_thr),
            "cv_best_f1": float(gs.best_score_),
            "best_params": gs.best_params_,
            # train
            "train_acc": tr_m["acc"], "train_precision": tr_m["precision"],
            "train_recall": tr_m["recall"], "train_f1": tr_m["f1"], "train_pos_rate": tr_m["pos_rate"],
            # test
            "test_acc": te_m["acc"], "test_precision": te_m["precision"],
            "test_recall": te_m["recall"], "test_f1": te_m["f1"], "test_pos_rate": te_m["pos_rate"],
            "model_obj": gs
        }
        

        key = (row["cv_best_f1"], row["test_f1"])
        if (best_game is None) or (key > (best_game["cv_best_f1"], best_game["test_f1"])):
            best_game = row
    
    


    if best_game is not None:
        tuned_rows.append(best_game)

        appid_int = best_game["appid"]
        best_model = best_game["model_obj"].best_estimator_


    path = f"model/model_{appid_int}.pkl"  # 동적 파일명

    joblib.dump(best_model, path)
    
    # result_list 초기화
    result_list = []

tuned_df = pd.DataFrame(tuned_rows).sort_values("test_f1", ascending=False).reset_index(drop=True)
print("튜닝 완료 게임 수:", len(tuned_df))
display(tuned_df.head(30))


FEATURES 개수: 30


Tuning games:   0%|          | 0/12 [00:00<?, ?it/s]

[LightGBM] [Info] Number of positive: 2321, number of negative: 25237
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003796 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1060
[LightGBM] [Info] Number of data points in the train set: 27558, number of used features: 26
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


Tuning games:   8%|▊         | 1/12 [02:44<30:06, 164.19s/it]

[LightGBM] [Info] Number of positive: 32240, number of negative: 142142
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.017018 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1066
[LightGBM] [Info] Number of data points in the train set: 174382, number of used features: 27
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


...

Tuning games:  92%|█████████▏| 11/12 [52:35<06:56, 416.12s/it]

[LightGBM] [Info] Number of positive: 47938, number of negative: 170062
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.046155 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1066
[LightGBM] [Info] Number of data points in the train set: 218000, number of used features: 27
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


Tuning games: 100%|██████████| 12/12 [1:04:02<00:00, 320.20s/it]


튜닝 완료 게임 수: 12


,appid,n_rows,churn_rate,model,best_thr,cv_best_f1,best_params,train_acc,train_precision,train_recall,train_f1,train_pos_rate,test_acc,test_precision,test_recall,test_f1,test_pos_rate,model_obj
0,3513350,37631,0.179878,LightGBM,0.56,0.422907,"{'clf__colsample_bytree': 1.0, 'clf__learning_...",0.828926,0.523682,0.541090,0.532243,0.185856,0.794208,0.431770,0.455687,0.443406,0.189850,"GridSearchCV(cv=StratifiedKFold(n_splits=2, ra..."
1,1245620,108874,0.265821,Logistic,0.40,0.409302,{'clf__C': 1.0},0.359453,0.280706,0.902216,0.428189,0.854384,0.358760,0.280490,0.902384,0.427957,0.855155,"GridSearchCV(cv=StratifiedKFold(n_splits=2, ra..."
2,3241660,340625,0.219902,LightGBM,0.48,0.422620,"{'clf__colsample_bytree': 1.0, 'clf__learning_...",0.643725,0.334122,0.624585,0.435352,0.411068,0.636448,0.326119,0.612576,0.425639,0.413064,"GridSearchCV(cv=StratifiedKFold(n_splits=2, ra..."
3,294100,30831,0.233110,Logistic,0.44,0.398108,{'clf__C': 0.5},0.476889,0.275105,0.761002,0.404120,0.644786,0.479488,0.272937,0.740612,0.398876,0.632723,"GridSearchCV(cv=StratifiedKFold(n_splits=2, ra..."
4,227300,92151,0.227171,Logistic,0.46,0.385609,{'clf__C': 2.0},0.510296,0.275590,0.709620,0.397000,0.584943,0.509685,0.274754,0.706472,0.395640,0.584125,"GridSearchCV(cv=StratifiedKFold(n_splits=2, ra..."
5,3240220,138626,0.219338,LightGBM,0.48,0.387348,"{'clf__colsample_bytree': 0.8, 'clf__learning_...",0.618124,0.319284,0.654594,0.429215,0.449693,0.596119,0.292414,0.592666,0.391611,0.444529,"GridSearchCV(cv=StratifiedKFold(n_splits=2, ra..."
6,1808500,252277,0.186660,LightGBM,0.50,0.382335,"{'clf__colsample_bytree': 0.8, 'clf__learning_...",0.700765,0.321282,0.542100,0.403453,0.314952,0.693753,0.308590,0.516458,0.386338,0.312391,"GridSearchCV(cv=StratifiedKFold(n_splits=2, ra..."
7,1551360,53036,0.217230,Logistic,0.48,0.381258,{'clf__C': 0.5},0.588691,0.285349,0.593794,0.385463,0.452060,0.590309,0.286133,0.592882,0.385985,0.450038,"GridSearchCV(cv=StratifiedKFold(n_splits=2, ra..."
8,1973530,43061,0.084206,LightGBM,0.68,0.315266,"{'clf__colsample_bytree': 0.8, 'clf__learning_...",0.890676,0.391250,0.536367,0.452457,0.115449,0.878323,0.332295,0.441379,0.379147,0.111808,"GridSearchCV(cv=StratifiedKFold(n_splits=2, ra..."
9,553850,148012,0.139239,LightGBM,0.58,0.349233,"{'clf__colsample_bytree': 0.8, 'clf__learning_...",0.798191,0.341614,0.484624,0.400742,0.197527,0.786745,0.310500,0.435468,0.362516,0.195284,"GridSearchCV(cv=StratifiedKFold(n_splits=2, ra..."


In [28]:
# tuned_df는 이미 "게임당 1개 최종(best_game)"만 들어있는 상태라고 가정
final_plan_df = tuned_df.copy()

final_plan_df = final_plan_df.rename(columns={
    "model": "best_model",
    "best_thr": "best_threshold",
    "test_f1": "best_f1",
    "best_params": "best_params",
})

# 저장/후처리용 필드 추가
final_plan_df["use_smote"] = False

# 필요한 컬럼만 깔끔하게
final_plan_df = final_plan_df[[
    "appid",
    "best_model",
    "best_params",
    "best_threshold",
    "best_f1",
    "use_smote",
    # 참고로 같이 두면 좋은 메타
    "n_rows",
    "churn_rate",
    "cv_best_f1",
    "test_acc", "test_precision", "test_recall", "test_pos_rate",
]]

display(final_plan_df)


,appid,best_model,best_params,best_threshold,best_f1,use_smote,n_rows,churn_rate,cv_best_f1,test_acc,test_precision,test_recall,test_pos_rate
0,3513350,LightGBM,"{'clf__colsample_bytree': 1.0, 'clf__learning_...",0.56,0.443406,False,37631,0.179878,0.422907,0.794208,0.431770,0.455687,0.189850
1,1245620,Logistic,{'clf__C': 1.0},0.40,0.427957,False,108874,0.265821,0.409302,0.358760,0.280490,0.902384,0.855155
2,3241660,LightGBM,"{'clf__colsample_bytree': 1.0, 'clf__learning_...",0.48,0.425639,False,340625,0.219902,0.422620,0.636448,0.326119,0.612576,0.413064
3,294100,Logistic,{'clf__C': 0.5},0.44,0.398876,False,30831,0.233110,0.398108,0.479488,0.272937,0.740612,0.632723
4,227300,Logistic,{'clf__C': 2.0},0.46,0.395640,False,92151,0.227171,0.385609,0.509685,0.274754,0.706472,0.584125
5,3240220,LightGBM,"{'clf__colsample_bytree': 0.8, 'clf__learning_...",0.48,0.391611,False,138626,0.219338,0.387348,0.596119,0.292414,0.592666,0.444529
6,1808500,LightGBM,"{'clf__colsample_bytree': 0.8, 'clf__learning_...",0.50,0.386338,False,252277,0.186660,0.382335,0.693753,0.308590,0.516458,0.312391
7,1551360,Logistic,{'clf__C': 0.5},0.48,0.385985,False,53036,0.217230,0.381258,0.590309,0.286133,0.592882,0.450038
8,1973530,LightGBM,"{'clf__colsample_bytree': 0.8, 'clf__learning_...",0.68,0.379147,False,43061,0.084206,0.315266,0.878323,0.332295,0.441379,0.111808
9,553850,LightGBM,"{'clf__colsample_bytree': 0.8, 'clf__learning_...",0.58,0.362516,False,148012,0.139239,0.349233,0.786745,0.310500,0.435468,0.195284


In [29]:
show_cols = [
    "appid","n_rows","churn_rate","model","best_thr","cv_best_f1",
    "train_acc","train_precision","train_recall","train_f1","train_pos_rate",
    "test_acc","test_precision","test_recall","test_f1","test_pos_rate"
]

out = tuned_df.copy()
for c in out.columns:
    if out[c].dtype.kind in "fc":
        out[c] = out[c].round(6)

display(out[show_cols])


,appid,n_rows,churn_rate,model,best_thr,cv_best_f1,train_acc,train_precision,train_recall,train_f1,train_pos_rate,test_acc,test_precision,test_recall,test_f1,test_pos_rate
0,3513350,37631,0.179878,LightGBM,0.56,0.422907,0.828926,0.523682,0.541090,0.532243,0.185856,0.794208,0.431770,0.455687,0.443406,0.189850
1,1245620,108874,0.265821,Logistic,0.40,0.409302,0.359453,0.280706,0.902216,0.428189,0.854384,0.358760,0.280490,0.902384,0.427957,0.855155
2,3241660,340625,0.219902,LightGBM,0.48,0.422620,0.643725,0.334122,0.624585,0.435352,0.411068,0.636448,0.326119,0.612576,0.425639,0.413064
3,294100,30831,0.233110,Logistic,0.44,0.398108,0.476889,0.275105,0.761002,0.404120,0.644786,0.479488,0.272937,0.740612,0.398876,0.632723
4,227300,92151,0.227171,Logistic,0.46,0.385609,0.510296,0.275590,0.709620,0.397000,0.584943,0.509685,0.274754,0.706472,0.395640,0.584125
5,3240220,138626,0.219338,LightGBM,0.48,0.387348,0.618124,0.319284,0.654594,0.429215,0.449693,0.596119,0.292414,0.592666,0.391611,0.444529
6,1808500,252277,0.186660,LightGBM,0.50,0.382335,0.700765,0.321282,0.542100,0.403453,0.314952,0.693753,0.308590,0.516458,0.386338,0.312391
7,1551360,53036,0.217230,Logistic,0.48,0.381258,0.588691,0.285349,0.593794,0.385463,0.452060,0.590309,0.286133,0.592882,0.385985,0.450038
8,1973530,43061,0.084206,LightGBM,0.68,0.315266,0.890676,0.391250,0.536367,0.452457,0.115449,0.878323,0.332295,0.441379,0.379147,0.111808
9,553850,148012,0.139239,LightGBM,0.58,0.349233,0.798191,0.341614,0.484624,0.400742,0.197527,0.786745,0.310500,0.435468,0.362516,0.195284


In [30]:
F1_CUT   = 0.40
RATE_CUT = 0.20

smote_targets_df = final_plan_df[
    (final_plan_df["churn_rate"] < RATE_CUT) &
    (final_plan_df["best_f1"] < F1_CUT)
].copy().sort_values(["churn_rate","best_f1"])

smote_appids = smote_targets_df["appid"].tolist()

print("SMOTE 대상 게임 수:", len(smote_appids))
print("SMOTE 대상 appid:", smote_appids)
display(smote_targets_df[["appid","churn_rate","best_model","best_f1","best_threshold","best_params"]])0.345242

SyntaxError: invalid syntax (808094398.py, line 13)

In [ ]:
# 필요하면 설치 (이미 되면 넘어감)
# !pip -q install imbalanced-learn

from tqdm import tqdm
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 42
TEST_SIZE = 0.20
VAL_SIZE_IN_TRAIN = 0.20

THR_GRID = np.round(np.arange(0.10, 0.91, 0.02), 2)
CV_SPLITS = 2
N_JOBS_GRID = 4

TARGET = "churn"

def get_proba(clf, X):
    return clf.predict_proba(X)[:, 1]

def metrics(y_true, y_pred):
    return {
        "acc": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "pos_rate": float(np.mean(y_pred == 1))
    }

def best_threshold_by_f1(y_true, proba, thr_grid=THR_GRID):
    best_f1, best_thr = -1.0, 0.5
    for thr in thr_grid:
        pred = (proba >= thr).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, float(thr)
    return best_thr

cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

smote_rows = []

for appid in tqdm(smote_appids, total=len(smote_appids), desc="SMOTE tuning games"):
    gdf = df_model[df_model["appid"] == appid].copy()
    gdf = gdf.replace([np.inf, -np.inf], 0)

    if gdf[TARGET].nunique() < 2:
        continue

    X = gdf[FEATURES]
    y = gdf[TARGET].astype(int)

    # SMOTE 과생성 방지: 매우 불균형이면 더 보수적으로
    churn_rate = float(y.mean())
    smote_ratio = 0.30 if churn_rate < 0.10 else 0.50
    smote = SMOTE(sampling_strategy=smote_ratio, random_state=RANDOM_STATE, k_neighbors=5)

    # SMOTE 포함 후보들 (grid는 너가 쓰던 "최소 튜닝" 그대로)
    candidates_smote = [
        ("Logistic",
         ImbPipeline([
             ("scaler", StandardScaler()),
             ("smote", smote),
             ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))
         ]),
         {"clf__C": [0.5, 1.0, 2.0]}),

        ("RandomForest",
         ImbPipeline([
             ("smote", smote),
             ("clf", RandomForestClassifier(
                 n_estimators=300, random_state=RANDOM_STATE, n_jobs=1, class_weight="balanced"
             ))
         ]),
         {"clf__max_depth": [None, 8],
          "clf__min_samples_leaf": [1, 20],
          "clf__max_features": ["sqrt"]}),

        ("HistGB",
         ImbPipeline([
             ("smote", smote),
             ("clf", HistGradientBoostingClassifier(random_state=RANDOM_STATE))
         ]),
         {"clf__learning_rate": [0.05, 0.1],
          "clf__max_leaf_nodes": [31, 63],
          "clf__min_samples_leaf": [20, 50]})
    ]

    # split
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=VAL_SIZE_IN_TRAIN,
        random_state=RANDOM_STATE, stratify=y_train_full
    )

    best_game = None

    for name, pipe, grid in candidates_smote:
        gs = GridSearchCV(pipe, grid, scoring="f1", cv=cv, n_jobs=N_JOBS_GRID, refit=True)
        gs.fit(X_train, y_train)

        best_est = gs.best_estimator_

        # threshold는 val에서 최적화
        val_proba = get_proba(best_est, X_val)
        best_thr = best_threshold_by_f1(y_val, val_proba)

        # train/test 평가
        tr_proba = get_proba(best_est, X_train_full)
        te_proba = get_proba(best_est, X_test)

        tr_pred = (tr_proba >= best_thr).astype(int)
        te_pred = (te_proba >= best_thr).astype(int)

        tr_m = metrics(y_train_full, tr_pred)
        te_m = metrics(y_test, te_pred)

        row = {
            "appid": int(appid),
            "n_rows": int(len(gdf)),
            "churn_rate": float(y.mean()),
            "best_model": name,
            "best_threshold": float(best_thr),
            "cv_best_f1": float(gs.best_score_),
            "best_params": gs.best_params_,
            "use_smote": True,
            "smote_sampling_strategy": smote_ratio,

            # train
            "train_acc": tr_m["acc"], "train_precision": tr_m["precision"],
            "train_recall": tr_m["recall"], "train_f1": tr_m["f1"], "train_pos_rate": tr_m["pos_rate"],
            # test
            "test_acc": te_m["acc"], "test_precision": te_m["precision"],
            "test_recall": te_m["recall"], "test_f1": te_m["f1"], "test_pos_rate": te_m["pos_rate"],
        }

        key = (row["cv_best_f1"], row["test_f1"])
        if (best_game is None) or (key > (best_game["cv_best_f1"], best_game["test_f1"])):
            best_game = row

    if best_game is not None:
        smote_rows.append(best_game)

        appid_int = best_game["appid"]
        best_model = best_game["model_obj"].best_estimator_


    path = f"model/model_{appid_int}.pkl"  # 동적 파일명

    joblib.dump(best_model, path)

smote_df = pd.DataFrame(smote_rows).sort_values("test_f1", ascending=False).reset_index(drop=True)
print("SMOTE 튜닝 완료 게임 수:", len(smote_df))
display(smote_df)


# 3. EDA 및 그래프 시각화